# KLA restoration training (bigger corpus)

Trains the residual U-Net on a **first-party synthetic, source-disjoint** corpus of 160 clean sources (768 train / 96 val / 96 test paired views). It does not use the supplied Drift-Sense Space (no competition-use licence observed). Results are pipeline evidence only, not official KLA or hidden-test scores.

This run produces `models/best.pth`, the exact checkpoint the `run.py` `.npy` evaluator loads. The I/O contract in `run.py` is unchanged.

Before running: **Runtime > Change runtime type > T4 GPU**, then run cells in order.

## 1. Record the actual runtime

In [ ]:
import platform, torch
print({
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'cuda': torch.version.cuda,
})
!nvidia-smi

## 2. Upload the current repository archive

From the local repository run `git archive --format=zip --output=kla_restoration_submission.zip HEAD`, then upload that ZIP here. This avoids relying on an unpushed private GitHub branch.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'kla_restoration_submission.zip' in uploaded, 'Upload kla_restoration_submission.zip'
!rm -rf kla-image-restoration
!unzip -q kla_restoration_submission.zip -d kla-image-restoration
%cd kla-image-restoration
!git rev-parse --short HEAD || true
!ls

## 3. Install the declared dependencies

In [ ]:
!python -m pip install -q -r requirements.txt
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 4. Construct the disclosed first-party corpus

The source split occurs by SHA-256 **before** views. The corpus uses only Gaussian noise, multiplicative speckle and downsampling, across all six orders.

In [ ]:
!python scripts/generate_clean_sem_sources.py --out data/sources_big --count 160 --size 768 --seed 20260817
!python scripts/materialize_restoration_data.py --source-dir data/sources_big --out data/kla_big --seed 20260817 --views-per-source 6 --crop-size 512 --scale 2
!cat data/kla_big/dataset_card.json

## 5. Train the bigger configuration

Model is selected by validation PSNR only. Do not inspect or tune against `data/kla_big/test`. On a T4 this is roughly 60 epochs of 768 samples; expect ~20-40 min.

In [ ]:
!python train.py --config configs/submission_big.yaml
!ls -la runs/kla_restoration_big_seed20260817

## 6. Freeze validation-best weights into the run.py location and evaluate held-out sources once

In [ ]:
!mkdir -p models weights results/submission_big/examples
# models/best.pth is the exact file run.py loads for the .npy evaluator contract.
!cp runs/kla_restoration_big_seed20260817/best.pth models/best.pth
!cp runs/kla_restoration_big_seed20260817/best.pth weights/final_model.pth
!cp runs/kla_restoration_big_seed20260817/resolved_config.yaml weights/final_model.config.yaml
!sha256sum models/best.pth | tee models/best.sha256
!python evaluate.py --gt-dir data/kla_big/test/GT --noisy-dir data/kla_big/test/NoisyLR --checkpoint models/best.pth --output-dir results/submission_big --save-restored results/submission_big/examples --split all --eval-mode official
!cat results/submission_big/summary.json

## 7. Exercise the evaluator-facing run.py `.npy` contract

In [ ]:
!rm -rf submission_smoke
# run.py takes positional <input-dir> <output-dir>, reads .npy recursively, writes matching .npy outputs.
!python run.py data/kla_big/test/NoisyLR submission_smoke
import numpy as np, glob, os
outs = sorted(glob.glob('submission_smoke/**/*.npy', recursive=True))
ins = sorted(glob.glob('data/kla_big/test/NoisyLR/**/*.npy', recursive=True))
print('inputs', len(ins), 'outputs', len(outs))
a = np.load(outs[0])
print('sample out', os.path.basename(outs[0]), a.shape, a.dtype, float(a.min()), float(a.max()), 'finite', bool(np.isfinite(a).all()))
assert len(ins) == len(outs), 'filename parity failed'

## 8. Archive artifacts and the trained checkpoint

Download this archive. It contains `models/best.pth` (drop it into the repo at the same path to update the submission), the evaluation summary, and manifests.

In [ ]:
import hashlib, pathlib, datetime
artifact = pathlib.Path('kla_restoration_big_artifacts.zip')
!zip -qr $artifact models/best.pth models/best.sha256 weights results/submission_big submission_smoke data/kla_big/dataset_card.json data/kla_big/train_manifest.csv data/kla_big/val_manifest.csv data/kla_big/test_manifest.csv
print({'artifact': str(artifact), 'sha256': hashlib.sha256(artifact.read_bytes()).hexdigest(), 'utc_finished': datetime.datetime.now(datetime.UTC).isoformat()})
from google.colab import files
files.download(str(artifact))